# 🐍 Master Guide: Context Managers & The `with` Statement
> **Single-Source Revision Guide**: Covers deterministic resource cleanup, `__enter__` and `__exit__` protocols, exception handling in context managers, and `contextlib`.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Context managers guarantee setup and teardown actions—such as closing file handles, releasing threading locks, or restoring database transactions—even if exceptions occur.

---
## 🧠 2. CPython Architecture: How `with` Translates to Bytecode

### The Exact Mechanics of the `with` Statement
```python
with Manager() as target:
    suite()
```
CPython compiles this into equivalent `try...except...finally` bytecode:

```python
mgr = Manager()
exit_fn = type(mgr).__exit__
value = type(mgr).__enter__(mgr)
hit_exception = False

try:
    target = value
    suite()
except Exception as e:
    hit_exception = True
    # If __exit__ returns True, exception is suppressed!
    if not exit_fn(mgr, *sys.exc_info()):
        raise
finally:
    if not hit_exception:
        exit_fn(mgr, None, None, None)
```

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Class-Based Context Manager (`__enter__` and `__exit__`)

In [ ]:
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
    
    def __enter__(self):
        print(f"[Setup] Opening connection to {self.db_name}...")
        return f"Session({self.db_name})"
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[Teardown] Closing connection to {self.db_name}.")
        # Return False so any exceptions propagate normally:
        return False

with DatabaseConnection("Production_DB") as session:
    print("Inside with block using session:", session)

---
## 🟡 4. Part 2: Intermediate Features

### Generator Context Managers with `@contextlib.contextmanager`
Write context managers with simple generator functions instead of entire classes!

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def benchmark_timer(task_name):
    print(f"Starting: {task_name}")
    start_time = time.perf_counter()
    try:
        yield  # Execution yields back to the 'with' body
    finally:
        elapsed = time.perf_counter() - start_time
        print(f"Finished: {task_name} in {elapsed*1000:.3f} ms")

with benchmark_timer("Summing 1 Million Integers"):
    _ = sum(range(1_000_000))

### Helpful Utilities: `contextlib.suppress` and `ExitStack`
- `suppress(Exception)`: Silently catches specified exceptions.
- `ExitStack`: Dynamically manages an arbitrary number of context managers.

In [ ]:
from contextlib import suppress
import os

# Instead of try...except FileNotFoundError: pass
with suppress(FileNotFoundError):
    os.remove("non_existent_file.tmp")

print("Suppressed FileNotFoundError safely without crashing!")

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Accidental Exception Suppression Trap
If `__exit__` returns `True`, Python **swallows the exception silently**!

In [ ]:
class DangerousManager:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        # BUG: Returning True suppresses ANY exception that occurred in with-block!
        return True

with DangerousManager():
    # This zero division error will be completely hidden!
    1 / 0

print("Notice: ZeroDivisionError was silently swallowed! (Dangerous in production)")

### ⚠️ Gotcha 2: Generator `@contextmanager` Exception Handling
In `@contextmanager`, if an exception happens in the `with` block, it is raised **at the `yield` statement**! You must wrap `yield` in a `try...finally`.

In [ ]:
print("Rule: In @contextmanager functions, always wrap yield in try...finally to guarantee cleanup!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| __enter__(self)                    | Sets up resource, returns target value|
| __exit__(self, exc_type, val, tb)  | Cleans up; return False to raise exc  |
| __exit__ returning True            | SWALLOWS/SUPPRESSES the exception!    |
| @contextlib.contextmanager         | Generator pattern: code before/after  |
| contextlib.suppress(Exc)           | Replaces try...except Exc: pass       |
| ExitStack                          | Manages dynamic list of resources     |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Robust Context Management & Automatic Resource Cleanup
Demonstrating how `with` statements ensure file handles, locks, or network connections close even during exceptions.

In [ ]:
# Custom Resource Context Manager implementing __enter__ and __exit__
class DatabaseConnectionSimulation:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connected = False

    def __enter__(self):
        self.connected = True
        print(f"[DB: {self.db_name}] Connection OPENED successfully.")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.connected = False
        print(f"[DB: {self.db_name}] Connection CLOSED cleanly in __exit__.")
        if exc_type is not None:
            print(f"  Handled exception gracefully inside context: {exc_val}")
            return True  # Suppress exception if desired

    def execute_query(self, query):
        if not self.connected:
            raise ConnectionError("Cannot execute query: Database not connected!")
        print(f"  Executing: '{query}' -> Query executed successfully.")

# Using the context manager
with DatabaseConnectionSimulation("Production_Oracle_DB") as db:
    db.execute_query("SELECT * FROM STUDENTS WHERE MARKS > 75")
    # Triggering an exception inside context
    raise ValueError("Simulated network timeout during transaction")

print("Execution continued smoothly after context manager handled cleanup.")